# AION-C · MoSE Tiny — Training Notebook

**Mixture of Specialized Engines — entrenamiento end-to-end completamente inline.**

| Fase | Steps | Datos |
|------|-------|-------|
| Shared pretraining | 1 000 | 5 dominios mezclados (motor random por batch) |
| Per-motor fine-tune × 5 | 500 | Dataset específico del dominio |

**Eval después de cada motor:** 3 ejemplos con Word F1.

Requisitos: T4 GPU · PyTorch ≥ 2.0 · < 10 min total.

> **Escalabilidad a 350M:** cambiar únicamente `MoSEConfig.medium_350m()`.
> Las interfaces de Encoder, Decoder, Motor, Orchestrator y Unifier son idénticas.

In [ ]:
# ── 1. Imports · Device · Config ────────────────────────────────────────────────
import os, sys, time, math, random
from dataclasses import dataclass
from typing import List, Dict, Optional
import torch, torch.nn as nn, torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler

if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime → Change runtime type → T4 GPU')
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
_f, _t = torch.cuda.mem_get_info()
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {_f/1e9:.1f}/{_t/1e9:.1f} GB libre')
torch.manual_seed(42); random.seed(42)


@dataclass
class MoSEConfig:
    """
    Config unificada para todos los componentes MoSE.
    Escalar a ~350M: usar MoSEConfig.medium_350m().
    Todas las interfaces (Encoder, Decoder, Motor, Orchestrator, Unifier)
    son idénticas entre tiny y 350M — solo cambian los valores de cfg.

      Tiny : hidden=128  heads=4   layers=2  vocab=128   seq=64   nodes=8
      350M : hidden=1024 heads=16  layers=24 vocab=32000 seq=2048 nodes=32
    """
    hidden_dim:          int   = 128   # → 1024
    vocab_size:          int   = 128   # → 32000 (BPE)
    n_enc_layers:        int   = 2     # → 24
    n_dec_layers:        int   = 2     # → 24
    n_heads:             int   = 4     # → 16
    ffn_mult:            int   = 4
    max_seq_len:         int   = 64    # → 2048
    max_graph_nodes:     int   = 8     # → 32
    cre_n_iters:         int   = 2     # → 5
    max_active_motors:   int   = 2
    orch_min_confidence: float = 0.30
    dropout:             float = 0.0   # → 0.1

    def __post_init__(self):
        assert self.hidden_dim % self.n_heads == 0

    @classmethod
    def tiny(cls): return cls()

    @classmethod
    def medium_350m(cls):
        """Mismas interfaces Python — solo cambian los valores."""
        return cls(hidden_dim=1024, vocab_size=32000, n_enc_layers=24,
                   n_dec_layers=24, n_heads=16, max_seq_len=2048,
                   max_graph_nodes=32, cre_n_iters=5, dropout=0.1)


cfg = MoSEConfig.tiny()
print(f'\nConfig tiny: D={cfg.hidden_dim} V={cfg.vocab_size} '
      f'L={cfg.max_seq_len} K={cfg.max_graph_nodes} '
      f'enc={cfg.n_enc_layers}L dec={cfg.n_dec_layers}L')

In [ ]:
# ── 2. Tokenizer + Data Generators (Levels 1-2) ──────────────────────────
# Tokenizador ASCII char-level: ord(c) para printable ASCII (32-126).
# Tokens especiales: PAD=0 BOS=1 EOS=2.
# Scale-up: reemplazar encode/decode por BPE — interfaz sin cambios.

PAD, BOS, EOS = 0, 1, 2

def encode(text: str, add_eos=True) -> List[int]:
    ids = [BOS] + [ord(c) for c in text if 32 <= ord(c) < cfg.vocab_size]
    if add_eos: ids.append(EOS)
    return ids

def decode(ids) -> str:
    return ''.join(chr(i) for i in ids if 32 <= i < 127)

def make_batch(texts: List[str], device) -> torch.Tensor:
    seqs    = [encode(t) for t in texts]
    max_len = min(max(len(s) for s in seqs), cfg.max_seq_len)
    padded  = [s[:max_len] + [PAD] * (max_len - len(s[:max_len])) for s in seqs]
    return torch.tensor(padded, dtype=torch.long, device=device)

assert decode(encode('hello world 42')) == 'hello world 42', 'tokenizer fail'
print('Tokenizer OK')

# ── Generadores (niveles 1-2 solamente) ────────────────────────────────────
_r = random.Random(99)

# 2a. CORA — Razonamiento causal
_PAIRS = [('rain','wet ground'),('fire','smoke'),('cold','ice'),
          ('stress','headache'),('virus','fever'),('exercise','strength'),
          ('sun','heat'),('wind','waves'),('flood','damage'),('drought','fire')]

def gen_causal() -> str:
    lvl = _r.randint(1, 2)
    if lvl == 1:
        a, b = _r.choice(_PAIRS)
        return f'{a} causes {b}. does {a} cause {b}? yes. {a} directly causes {b}.'
    a, b = _r.choice(_PAIRS)
    bcs  = [p for p in _PAIRS if p[0] == b]
    c    = _r.choice(bcs)[1] if bcs else _r.choice(_PAIRS)[1]
    if c == a: c = 'harm'
    return f'{a} causes {b}. {b} causes {c}. does {a} cause {c}? yes. {a} causes {c} via {b}.'

# 2b. FORGE-C — Dependencias de código
_FN = ['getData','parse','validate','render','save','load','process','build','check','fetch']

def gen_code() -> str:
    lvl  = _r.randint(1, 2)
    a, b = _r.sample(_FN, 2)
    if lvl == 1:
        return f'{a} calls {b}. if {b} changes, what is affected? {a} is affected.'
    c = _r.choice([f for f in _FN if f not in (a, b)])
    return f'{a} calls {b}. {b} calls {c}. if {c} changes, what breaks? {b} breaks. {a} breaks.'

# 2c. AXIOM — Transitividad matemática
_V = list('abcdefgh')

def gen_math() -> str:
    lvl = _r.randint(1, 2)
    if lvl == 1:
        x, y   = _r.sample(_V[:6], 2)
        n1, n2 = sorted(_r.sample(range(1, 20), 2), reverse=True)
        return f'{x} = {n1}. {y} = {n2}. is {x} > {y}? yes. {x} > {y} since {n1} > {n2}.'
    x, y, z = _r.sample(_V[:6], 3)
    return f'{x} > {y}. {y} > {z}. is {x} > {z}? yes. {x} > {z} by transitivity.'

# 2d. MUSE — Narrativa de conflicto
_HERO = ['hero','knight','rebel','wanderer','mage']
_GOAL = ['freedom','justice','treasure','peace','truth']
_VILL = ['villain','tyrant','shadow','guard','dragon']

def gen_narrative() -> str:
    lvl     = _r.randint(1, 2)
    hero    = _r.choice(_HERO)
    goal    = _r.choice(_GOAL)
    if lvl == 1:
        return f'{hero} wants {goal}. what drives {hero}? {hero} is driven by desire for {goal}.'
    villain = _r.choice(_VILL)
    return (f'{hero} seeks {goal}. {villain} blocks {hero}. '
            f'how is conflict resolved? '
            f'{hero} overcomes {villain} and achieves {goal}.')

# 2e. EMPATHY — Comprensión social
_NAME  = ['Ana','Bob','Carl','Dana','Eve','Frank']
_STATE = ['happy','sad','angry','tired','scared','confused']

def gen_social() -> str:
    lvl = _r.randint(1, 2)
    a, b = _r.sample(_NAME, 2)
    if lvl == 1:
        real = _r.choice(_STATE)
        bld  = _r.choice([s for s in _STATE if s != real])
        return (f'{a} thinks {b} is {bld}. {b} is really {real}. '
                f'is there a misunderstanding? '
                f'yes. {a} misread {b}. {b} is {real} not {bld}.')
    action = _r.choice(['arrived late','left early','stayed silent','changed plans'])
    expect = _r.choice(['be on time','stay until end','speak up','keep plans'])
    return (f'{a} expected {b} to {expect}. {b} {action}. '
            f'what happened? expectation violated. {b} {action}.')

# ── Registry ───────────────────────────────────────────────────────────────
MOTOR_NAMES = ['cora', 'forge_c', 'axiom', 'muse', 'empathy']
GENERATORS  = {'cora': gen_causal, 'forge_c': gen_code,
               'axiom': gen_math,  'muse': gen_narrative, 'empathy': gen_social}

def get_batch(domain: str, bs: int, device) -> torch.Tensor:
    return make_batch([GENERATORS[domain]() for _ in range(bs)], device)

def get_mixed_batch(bs: int, device) -> torch.Tensor:
    per = max(1, bs // len(MOTOR_NAMES))
    texts = []
    for d in MOTOR_NAMES: texts += [GENERATORS[d]() for _ in range(per)]
    _r.shuffle(texts)
    return make_batch(texts[:bs], device)

print('Data generators OK:')
for d, g in GENERATORS.items():
    s = g(); print(f'  [{d:<8}] {s[:72]}{"..." if len(s)>72 else ""}')

In [ ]:
# ── 3. Model Architecture (scale-invariant) ──────────────────────────────
# Todas las interfaces son idénticas para tiny y 350M.
# Solo cambia cfg. Ningún otro código necesita modificarse.

# ── StreamEncoder: token_ids [B,L] → enc_out [B,L,D] ──────────────────────
class StreamEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        D = cfg.hidden_dim
        self.embed = nn.Embedding(cfg.vocab_size, D, padding_idx=PAD)
        self.pos   = nn.Embedding(cfg.max_seq_len, D)
        layer      = nn.TransformerEncoderLayer(D, cfg.n_heads, D * cfg.ffn_mult,
                         dropout=cfg.dropout, batch_first=True, norm_first=True)
        self.enc   = nn.TransformerEncoder(layer, cfg.n_enc_layers)
        self.norm  = nn.LayerNorm(D)
        nn.init.normal_(self.embed.weight, std=0.02)

    def forward(self, x):                          # [B,L] → [B,L,D]
        B, L = x.shape
        h = self.embed(x) + self.pos(torch.arange(L, device=x.device))
        return self.norm(self.enc(h, src_key_padding_mask=x.eq(PAD)))


# ── StreamDecoder: (token_ids [B,L], graph [B,K,D]) → logits [B,L,V] ─────────
class StreamDecoder(nn.Module):
    def __init__(self, cfg, shared_embed):
        super().__init__()
        D = cfg.hidden_dim
        self.embed = shared_embed                          # weight-tied con encoder
        self.pos   = nn.Embedding(cfg.max_seq_len, D)
        layer      = nn.TransformerDecoderLayer(D, cfg.n_heads, D * cfg.ffn_mult,
                         dropout=cfg.dropout, batch_first=True, norm_first=True)
        self.dec   = nn.TransformerDecoder(layer, cfg.n_dec_layers)
        self.norm  = nn.LayerNorm(D)
        self.proj  = nn.Linear(D, cfg.vocab_size, bias=False)
        self.proj.weight = shared_embed.weight             # weight tying

    def forward(self, x, graph):                   # → [B,L,V]
        B, L = x.shape
        h    = self.embed(x) + self.pos(torch.arange(L, device=x.device))
        cm   = torch.triu(torch.full((L, L), float('-inf'), device=x.device), 1)
        h    = self.dec(h, graph, tgt_mask=cm, tgt_key_padding_mask=x.eq(PAD))
        return self.proj(self.norm(h))


# ── Motor: enc_out [B,L,D] → graph_repr [B,K,D] ───────────────────────────
class Motor(nn.Module):
    """
    Crystallizer (soft-attn pooling) + CRE (message passing GRU).
    Tiny: single shared msg_fn.
    350M: reemplazar self.msg por nn.ModuleList de n_relations funciones.
    La interfaz forward(enc_out) → [B,K,D] no cambia al escalar.
    """
    def __init__(self, cfg, name=''):
        super().__init__()
        D, K    = cfg.hidden_dim, cfg.max_graph_nodes
        self.K  = K; self.name = name
        self.gate = nn.Linear(D, K)
        self.proj = nn.Linear(D, D)
        self.msg  = nn.Linear(D * 2, D)
        self.gru  = nn.GRUCell(D, D)
        self.norm = nn.LayerNorm(D)

    def crystallize(self, enc):                    # [B,L,D] → [B,K,D]
        w = self.gate(enc).softmax(dim=1)          # [B,L,K]
        return self.proj(torch.einsum('blk,bld->bkd', w, enc))

    def cre(self, h, n):                           # [K,D] → [K,D]
        K, D = h.shape
        for _ in range(n):
            s   = h.unsqueeze(1).expand(K, K, D)
            t   = h.unsqueeze(0).expand(K, K, D)
            agg = self.msg(torch.cat([s, t], -1)).mean(1)
            h   = self.gru(agg, h)
        return self.norm(h)

    def forward(self, enc, n_iters=None):          # [B,L,D] → [B,K,D]
        n     = n_iters if n_iters is not None else 2
        nodes = self.crystallize(enc)
        return torch.stack([self.cre(nodes[b], n) for b in range(nodes.shape[0])])


# ── Orchestrator: enc_out → motor_names, scores, mode ─────────────────────
_KW = {
    'cora':    ['causes','cause','leads','prevents','implies'],
    'forge_c': ['calls','function','build','fetch','process','render','load','save',
               'breaks','affected','import','code','depends'],
    'axiom':   ['> ','transitivity','theorem','prove','= ','is a ','is b '],
    'muse':    ['hero','villain','conflict','story','seeks','wants','overcomes'],
    'empathy': ['thinks','misunderstanding','expected','misread','really','feels'],
}

class Orchestrator(nn.Module):
    def __init__(self, cfg, names):
        super().__init__()
        D = cfg.hidden_dim
        self.names    = names
        self.min_conf = cfg.orch_min_confidence
        self.max_act  = cfg.max_active_motors
        self.clf      = nn.Sequential(
            nn.Linear(D, D // 2), nn.GELU(), nn.LayerNorm(D // 2),
            nn.Linear(D // 2, len(names)))

    def forward(self, enc, text=None):
        scores = self.clf(enc.mean(1).mean(0)).softmax(-1)
        ranked = scores.argsort(descending=True).tolist()
        active = [ranked[0]]
        for i in ranked[1:self.max_act]:
            if scores[i] >= self.min_conf: active.append(i)
        mode = 'learned'
        if scores.max() < self.min_conf and text:
            t    = text.lower()
            hits = {m: sum(kw in t for kw in kws) for m, kws in _KW.items()}
            best = max(hits, key=hits.get) if any(hits.values()) else 'cora'
            active, mode = [self.names.index(best)], 'heuristic'
        return [self.names[i] for i in active], scores.detach(), mode


# ── Unifier: List[[K,D]] → [K,D] ───────────────────────────────────────────
class Unifier(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        D = cfg.hidden_dim; self.K = cfg.max_graph_nodes
        self.attn = nn.MultiheadAttention(D, cfg.n_heads, batch_first=True,
                                           dropout=cfg.dropout)
        self.mlp  = nn.Sequential(nn.LayerNorm(D), nn.Linear(D, D * 2),
                                   nn.GELU(), nn.Linear(D * 2, D), nn.LayerNorm(D))
        self.norm = nn.LayerNorm(D)

    def _pool(self, x):
        N, D = x.shape; K = self.K
        if N >= K: return x[x.norm(dim=-1).topk(K).indices.sort().values]
        return torch.cat([x, x.new_zeros(K - N, D)])

    def forward(self, reprs):                      # List[[K,D]] → [K,D]
        if len(reprs) == 1: return self._pool(reprs[0])
        cat  = torch.cat(reprs).unsqueeze(0)
        a, _ = self.attn(self.norm(cat), self.norm(cat), self.norm(cat))
        return self._pool(self.mlp(a.squeeze(0) + cat.squeeze(0)))


# ── MoSEPipeline ──────────────────────────────────────────────────────────────
class MoSEPipeline(nn.Module):
    """
    Encoder → Orchestrator → Motors → Unifier → Decoder.
    forward(x [B,L], active=None, text=None) → logits [B,L,V]
    Swap cfg = MoSEConfig.medium_350m() para escalar a ~350M.
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg    = cfg
        self.enc    = StreamEncoder(cfg)
        self.dec    = StreamDecoder(cfg, self.enc.embed)
        self.orch   = Orchestrator(cfg, MOTOR_NAMES)
        self.unif   = Unifier(cfg)
        self.motors = nn.ModuleDict({n: Motor(cfg, name=n) for n in MOTOR_NAMES})

    def forward(self, x, active=None, text=None):  # [B,L] → [B,L,V]
        B      = x.shape[0]
        enc    = self.enc(x)
        if active is None:
            active, _, _ = self.orch(enc, text)
        mouts  = [self.motors[m](enc) for m in active]  # List[[B,K,D]]
        graph  = torch.stack(
            [self.unif([mo[b] for mo in mouts]) for b in range(B)])
        return self.dec(x, graph)

    def n_params(self):
        seen = set(); n = 0
        for p in self.parameters():
            if id(p) not in seen: seen.add(id(p)); n += p.requires_grad * p.numel()
        return n

    def breakdown(self):
        pairs = [('encoder', self.enc), ('decoder', self.dec),
                 ('orchestrator', self.orch), ('unifier', self.unif)]
        pairs += [(n, m) for n, m in self.motors.items()]
        bd = {k: sum(p.numel() for p in m.parameters()) for k, m in pairs}
        bd['total_unique'] = self.n_params()
        return bd

In [ ]:
# ── 4. Build Pipeline + Phase 1: Shared Pretraining ───────────────────────
# 1000 steps, batch mixto de los 5 dominios.
# Motor seleccionado aleatoriamente por batch → todos los motores preentrenados.

BATCH = 64
LR    = 5e-4
STEPS = 1_000
LOG   = 100

pipeline = MoSEPipeline(cfg).to(DEVICE)
scaler   = GradScaler()
opt      = torch.optim.AdamW(pipeline.parameters(), lr=LR, weight_decay=0.01)
sched    = torch.optim.lr_scheduler.OneCycleLR(
               opt, LR, total_steps=STEPS, pct_start=0.1, anneal_strategy='cos')

# Parameter breakdown
bd = pipeline.breakdown()
print(f"{'Module':<15} {'Params':>10}")
print('-' * 28)
for k, v in bd.items(): print(f'  {k:<13} {v:>10,}')
print()

t_total = time.perf_counter()
losses  = []
pipeline.train()

for step in range(1, STEPS + 1):
    motor = random.choice(MOTOR_NAMES)
    batch = get_mixed_batch(BATCH, DEVICE)

    with autocast():
        logits = pipeline(batch, active=[motor])
        loss   = F.cross_entropy(
            logits[:, :-1].reshape(-1, cfg.vocab_size),
            batch[:, 1:].reshape(-1), ignore_index=PAD)

    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    nn.utils.clip_grad_norm_(pipeline.parameters(), 1.0)
    scaler.step(opt); scaler.update()
    opt.zero_grad(set_to_none=True)
    sched.step()
    losses.append(loss.item())

    if step % LOG == 0:
        avg = sum(losses[-LOG:]) / LOG
        lr  = sched.get_last_lr()[0]
        print(f'  step {step:>5}/{STEPS}  loss={avg:.4f}  lr={lr:.2e}  motor={motor}')

print(f'\nPhase 1 done | loss {losses[0]:.4f} → {sum(losses[-10:])/10:.4f}')
print(f'Time: {time.perf_counter()-t_total:.0f}s')

In [ ]:
# ── 5. Phase 2: Per-Motor Fine-Tuning + Eval ─────────────────────────────
# 500 steps por motor con datos del dominio específico.
# Motor params: LR 4x más alto que enc/dec compartidos.
# Eval: 3 ejemplos, Word F1.

LR_MOTOR  = 2e-4
LR_SHARED = 5e-5
STEPS_PER_MOTOR = {'cora': 2000, 'forge_c': 2000, 'axiom': 500, 'muse': 2000, 'empathy': 2000}
LOG_M     = 100

# ── Metrics ───────────────────────────────────────────────────────────────
def word_f1(pred: str, ref: str) -> float:
    p, r = set(pred.lower().split()), set(ref.lower().split())
    if not p or not r: return 0.0
    tp = len(p & r); pr = tp / len(p); rc = tp / len(r)
    return round(2 * pr * rc / (pr + rc), 3) if pr + rc else 0.0

@torch.no_grad()
def greedy_decode(prompt: str, motor: str, max_new=32) -> str:
    pipeline.eval()
    ids = encode(prompt, add_eos=False)[:cfg.max_seq_len - max_new]
    cur = torch.tensor([ids], device=DEVICE)
    for _ in range(max_new):
        if cur.shape[1] >= cfg.max_seq_len: break
        nxt = pipeline(cur, active=[motor])[0, -1].argmax().item()
        if nxt == EOS: break
        cur = torch.cat([cur, torch.tensor([[nxt]], device=DEVICE)], 1)
    return decode(cur[0, len(ids):].tolist())

def eval_motor(domain: str, n=3) -> float:
    f1s = []
    print(f'  Eval [{domain}]:')
    for i in range(n):
        full = GENERATORS[domain]()
        if '?' in full:
            q, a = full.split('?', 1); q += '?'; a = a.strip()
        else:
            mid = len(full) // 2; q, a = full[:mid], full[mid:].strip()
        if not a: continue
        pred = greedy_decode(q, domain, max_new=len(a) + 5)
        f1   = word_f1(pred, a)
        f1s.append(f1)
        print(f'    [{i+1}] Q: {q[:54]}{"..." if len(q)>54 else ""}')
        print(f'         A: {a[:54]}')
        print(f'         P: {pred[:54]}{"..." if len(pred)>54 else ""}  F1={f1:.2f}')
    mean = sum(f1s) / len(f1s) if f1s else 0.0
    print(f'         → mean Word F1 = {mean:.3f}')
    return mean

# ── Training loop ──────────────────────────────────────────────────────────────
motor_results: Dict[str, dict] = {}

_SEP = '━'
for domain in MOTOR_NAMES:
    steps_m = STEPS_PER_MOTOR[domain]
    print(f'\n{_SEP*56}')
    print(f'  Motor {domain.upper():<10}  [{steps_m} steps · data={domain}]')
    print(f'{_SEP*56}')

    mp = list(pipeline.motors[domain].parameters())
    sp = (list(pipeline.enc.parameters()) + list(pipeline.dec.parameters()) +
          list(pipeline.orch.parameters()) + list(pipeline.unif.parameters()))
    opt_m = torch.optim.AdamW([{'params': mp, 'lr': LR_MOTOR},
                                {'params': sp, 'lr': LR_SHARED}],
                               weight_decay=0.01)

    losses_m = []; t_m = time.perf_counter()
    pipeline.train()
    for step in range(1, steps_m + 1):
        # Linear warmup (50 steps) + cosine decay
        scale = min(1.0, step / 50) * (0.5 * (
            1 + math.cos(math.pi * max(0, step - 50) / max(1, steps_m - 50))))
        opt_m.param_groups[0]['lr'] = LR_MOTOR  * scale
        opt_m.param_groups[1]['lr'] = LR_SHARED * scale

        batch = get_batch(domain, BATCH, DEVICE)
        with autocast():
            logits = pipeline(batch, active=[domain])
            loss   = F.cross_entropy(
                logits[:, :-1].reshape(-1, cfg.vocab_size),
                batch[:, 1:].reshape(-1), ignore_index=PAD)

        scaler.scale(loss).backward()
        scaler.unscale_(opt_m)
        nn.utils.clip_grad_norm_(pipeline.parameters(), 1.0)
        scaler.step(opt_m); scaler.update()
        opt_m.zero_grad(set_to_none=True)
        losses_m.append(loss.item())

        if step % LOG_M == 0:
            avg = sum(losses_m[-LOG_M:]) / LOG_M
            print(f'    step {step:>4}/{steps_m}  loss={avg:.4f}  '
                  f't={time.perf_counter()-t_m:.0f}s')

    final = sum(losses_m[-50:]) / 50
    f1 = eval_motor(domain)
    motor_results[domain] = {'loss_start': losses_m[0], 'loss_end': final, 'f1': f1}
    print(f'  Done: {time.perf_counter()-t_m:.0f}s')

In [ ]:
# ── 6. Summary ────────────────────────────────────────────────────────────────
W = 58
_SEP = '━'
print(f'\n{_SEP*W}')
print('  MoSE Tiny — Training Summary')
print(f'{_SEP*W}')

# Routing correctness
TEST_Q = {
    'cora':    'rain causes flood. wet ground causes flood. does rain cause flood?',
    'forge_c': 'getData calls parse. parse calls validate. if validate code breaks, what happens?',
    'axiom':   'a > b. b > c. is a > c? use transitivity theorem.',
    'muse':    'hero seeks freedom. villain blocks hero. conflict resolved?',
    'empathy': 'Ana thinks Bob feels angry. Bob really feels sad. is there a misunderstanding?',
}
print('\n  Routing check (heuristic mode):')
print(f"  {'Domain':<10} {'Expected':<10} {'Got':<10} Match")
print('  ' + '-' * 36)
pipeline.eval()
with torch.no_grad():
    for domain, q in TEST_Q.items():
        x = make_batch([q], DEVICE)
        motors, _, mode = pipeline.orch(pipeline.enc(x), text=q)
        got   = motors[0]
        match = 'OK' if got == domain else f'FAIL->{got}'
        print(f'  {domain:<10} {domain:<10} {got:<10} {match}')

# Per-motor results table
print('\n  Per-motor training results:')
print(f"  {'Motor':<10} {'Loss start':>11} {'Loss final':>11} {'Word F1':>9}")
print('  ' + '-' * 46)
for d, r in motor_results.items():
    print(f"  {d:<10} {r['loss_start']:>11.4f} {r['loss_end']:>11.4f} {r['f1']:>9.3f}")

# Shared pretraining
p0 = sum(losses[:10]) / 10; p1 = sum(losses[-10:]) / 10
print(f'\n  Shared pretraining: {p0:.4f} → {p1:.4f}  (Δ={p0-p1:.4f})')

total = time.perf_counter() - t_total
print(f'  Total time : {total:.0f}s ({total/60:.1f} min)')
print(f'  Parameters : {pipeline.n_params():,}  '
      f'(D={cfg.hidden_dim}, enc={cfg.n_enc_layers}L, dec={cfg.n_dec_layers}L, K={cfg.max_graph_nodes})')
print(f'  Scale-up   : set cfg = MoSEConfig.medium_350m() — no other code changes')
print(f'{_SEP*W}')